# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [6]:
task_type = "Ranking / Scoring"

why = """
My lane is Refresh / Content Opportunity Scoring - the actual question is
"which pages need attention first?", which is a ranking/scoring problem,
not plain classification. The deliverable is an ordered priority queue
with scores, not a single yes/no label per page. A classification or
regression model can power the scoring underneath, but the TASK itself -
what the editor receives and acts on - is a ranked list, which is why it
falls under Ranking/Scoring rather than Classification.
"""
print(task_type)
print(why)

Ranking / Scoring

My lane is Refresh / Content Opportunity Scoring - the actual question is
"which pages need attention first?", which is a ranking/scoring problem,
not plain classification. The deliverable is an ordered priority queue
with scores, not a single yes/no label per page. A classification or
regression model can power the scoring underneath, but the TASK itself -
what the editor receives and acts on - is a ranked list, which is why it
falls under Ranking/Scoring rather than Classification.



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [7]:
target = "trend_pct"

target_explanation = """
I would predict trend_pct - the real, observed percentage change in a
page's performance over the measured trailing window. This label comes
from an OBSERVED outcome, not a rule I defined myself: it's a real,
measured value already present in the data, reflecting what actually
happened to the page's performance, not a proxy or something derived from
my own assumptions or a hand-picked formula.

I'm predicting the actual percentage (regression) rather than only a
yes/no decline flag, because severity varies meaningfully - the median
decline among falling pages is -55.6%, and the worst 10% have dropped 96%
or more. A binary label would flatten that difference and lose exactly
the information my lane needs to prioritize correctly. This also follows
Google's own ML framing guidance: since there's no fixed action threshold
here (editors simply work down a priority-ordered list, not a bucketed
decision), regression is the correct choice over classification.

Important: trend_pct and trend_direction are NEVER used as FEATURES -
only as the target - since trend_direction is itself computed from
trend_pct, and using either as an input would leak the answer directly
into the model.Because trend_pct is computed from recent traffic windows,
later feature engineering will explicitly check for target leakage by ensuring
that features do not directly encode the outcome being predicted.
"""
print(target)
print(target_explanation)

trend_pct

I would predict trend_pct - the real, observed percentage change in a
page's performance over the measured trailing window. This label comes
from an OBSERVED outcome, not a rule I defined myself: it's a real,
measured value already present in the data, reflecting what actually
happened to the page's performance, not a proxy or something derived from
my own assumptions or a hand-picked formula.

I'm predicting the actual percentage (regression) rather than only a
yes/no decline flag, because severity varies meaningfully - the median
decline among falling pages is -55.6%, and the worst 10% have dropped 96%
or more. A binary label would flatten that difference and lose exactly
the information my lane needs to prioritize correctly. This also follows
Google's own ML framing guidance: since there's no fixed action threshold
here (editors simply work down a priority-ordered list, not a bucketed
decision), regression is the correct choice over classification.

Important: trend_pct a

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [8]:
success_metric = "Precision@50"

success_metric_explanation = """
Precision@50: of the top 50 pages my model ranks as highest priority, what
percentage are genuinely declining (per real, held-out trend_pct data the
model never saw during training)?

What 'good' means: the reference pipeline for this lane establishes a
real benchmark - a naive baseline (sorting by trend_pct alone) scores
around 0.24 Precision@50. A trained model reaching approximately 0.74 or
higher would represent a genuine, defensible improvement - meaning
roughly 3 out of 4 flagged pages are actually declining, versus roughly
1 in 4 for the naive baseline.

I'm choosing Precision@50 specifically because 50 pages is a realistic
weekly review capacity for a single editor, tying the metric directly to
the actual action it supports.

Note: Precision@K checks membership (are the right pages in the top 50?)
but not order within it. I'll also track Spearman rank correlation later
in validation (ML-09) as a secondary check on ranking quality.
"""
print(success_metric)
print(success_metric_explanation)

Precision@50

Precision@50: of the top 50 pages my model ranks as highest priority, what
percentage are genuinely declining (per real, held-out trend_pct data the
model never saw during training)?

What 'good' means: the reference pipeline for this lane establishes a
real benchmark - a naive baseline (sorting by trend_pct alone) scores
around 0.24 Precision@50. A trained model reaching approximately 0.74 or
higher would represent a genuine, defensible improvement - meaning
roughly 3 out of 4 flagged pages are actually declining, versus roughly
1 in 4 for the naive baseline.

I'm choosing Precision@50 specifically because 50 pages is a realistic
weekly review capacity for a single editor, tying the metric directly to
the actual action it supports.

Note: Precision@K checks membership (are the right pages in the top 50?)
but not order within it. I'll also track Spearman rank correlation later
in validation (ML-09) as a secondary check on ranking quality.



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [9]:
unit_explanation = """
Unit of analysis: one row = one pseudonymized content page (content_id),
belonging to one pseudonymized client (client_id), with trailing-90-day
performance metrics (impressions, clicks, sessions, engagement), content
metadata (word count, content type, age), and a derived trend label
(trend_pct, trend_direction) describing that page's recent performance
change.

This is my lane's exact slice - the full starter dataset (30,000 rows,
44 columns) already matches the Refresh/Content Opportunity Scoring lane
directly, since every row is a candidate page for the ranked review
queue - no additional filtering was needed to isolate this lane's data.
"""
print(unit_explanation)


Unit of analysis: one row = one pseudonymized content page (content_id),
belonging to one pseudonymized client (client_id), with trailing-90-day
performance metrics (impressions, clicks, sessions, engagement), content
metadata (word count, content type, age), and a derived trend label
(trend_pct, trend_direction) describing that page's recent performance
change.

This is my lane's exact slice - the full starter dataset (30,000 rows,
44 columns) already matches the Refresh/Content Opportunity Scoring lane
directly, since every row is a candidate page for the ranked review
queue - no additional filtering was needed to isolate this lane's data.



## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [10]:
why_ml = """
A fixed rule (e.g., sort by trend_pct alone) already works for pages that
HAVE a measured trend_pct - no ML needed there, it's just sorting a real
number. But this rule has two real limits an if-statement can't fix:

1. New pages (7.5% of the dataset, trend_direction == 'new') have no
   trend_pct at all - there's nothing to sort. A model trained on
   pre-decline signals (CTR, engagement, content age) can still produce a
   priority estimate for these pages, where a fixed rule has nothing to
   work with.

2. Sorting trend_pct is reactive, not predictive - it only tells you what
   has ALREADY happened. The real value of ML is using PRE-decline signals
   (impressions_prev_30d, ctr, engagement_rate, content_age_days, etc.) to
   flag pages likely to decline before it's already visible in trend_pct.
   A single if-statement can't do this, because the actual DRIVER of
   decline differs page to page - sometimes CTR, sometimes position,
   sometimes engagement - and a fixed rule would need to hand-guess the
   right combination and weighting for every possible case. A model
   learns that combination directly from 30,000 real examples, and
   applies it per page, without me manually specifying the relationship
   between every feature and the outcome.
"""
print(why_ml)


A fixed rule (e.g., sort by trend_pct alone) already works for pages that
HAVE a measured trend_pct - no ML needed there, it's just sorting a real
number. But this rule has two real limits an if-statement can't fix:

1. New pages (7.5% of the dataset, trend_direction == 'new') have no
   trend_pct at all - there's nothing to sort. A model trained on
   pre-decline signals (CTR, engagement, content age) can still produce a
   priority estimate for these pages, where a fixed rule has nothing to
   work with.

2. Sorting trend_pct is reactive, not predictive - it only tells you what
   has ALREADY happened. The real value of ML is using PRE-decline signals
   (impressions_prev_30d, ctr, engagement_rate, content_age_days, etc.) to
   flag pages likely to decline before it's already visible in trend_pct.
   A single if-statement can't do this, because the actual DRIVER of
   decline differs page to page - sometimes CTR, sometimes position,
   sometimes engagement - and a fixed rule would n

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.